# SIGMA — Training segmentasi benih melon (baik / cacat)
Jalankan sel dari atas ke bawah. Sebelum mulai: **Runtime → Change runtime type → T4 GPU**.
Simpan API key Roboflow di panel **Secrets** (ikon kunci di kiri) dengan nama `ROBOFLOW_API_KEY`
dan aktifkan *Notebook access*. Jangan menempelkan API key langsung di dalam sel.

### 1. Cek GPU dan pasang pustaka

In [ ]:
!nvidia-smi -L
%pip install -q -U ultralytics roboflow

### 2. Sambungkan Google Drive (hasil training tidak hilang saat sesi putus)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SIMPAN = '/content/drive/MyDrive/sigma-benih/runs'

### 3. Unduh dataset dari Roboflow

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
import yaml

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
proyek = rf.workspace('sigmabenih').project('sigma-benih-melon')
ds = proyek.version(1).download('yolov8')   # format teks YOLO; dipakai juga oleh YOLO26/YOLO11

# Rapikan jalur di data.yaml supaya selalu terbaca, di mana pun folder diunduh
DATA = f'{ds.location}/data.yaml'
cfg = yaml.safe_load(open(DATA))
cfg.update(path=ds.location, train='train/images', val='valid/images', test='test/images')
yaml.safe_dump(cfg, open(DATA, 'w'))
NAMES = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
print(NAMES)

### 4. Hitung jumlah benih per split, varietas, dan kelas

In [ ]:
from pathlib import Path
from collections import Counter

for split in ['train', 'valid', 'test']:
    hitung = Counter()
    for f in Path(ds.location, split, 'labels').glob('*.txt'):
        var = f.name.split('_')[0]                       # VA / VB / VC dari nama berkas
        for baris in f.read_text().splitlines():
            if baris.strip():
                hitung[(var, NAMES[int(baris.split()[0])])] += 1
    print(f'{split:6s}', dict(sorted(hitung.items())))

### 5. Training

In [ ]:
from ultralytics import YOLO

try:
    model = YOLO('yolo26s-seg.pt')      # generasi terbaru Ultralytics
except Exception:
    model = YOLO('yolo11s-seg.pt')      # cadangan bila bobot YOLO26 tidak tersedia

model.train(
    data=DATA, epochs=150, patience=40, imgsz=1280, batch=8,   # 4 bila GPU kehabisan memori
    degrees=180, fliplr=0.5, flipud=0.5,   # benih boleh menghadap ke mana saja
    hsv_h=0.0, hsv_s=0.2, hsv_v=0.2,       # warna = ciri cacat, jangan digeser
    scale=0.1, translate=0.1,              # jarak kamera tetap, ukuran = ciri cacat
    mosaic=1.0, close_mosaic=20,
    project=SIMPAN, name='v1', exist_ok=True, seed=0, deterministic=True)

### 6. Evaluasi pada data test (belum pernah dilihat model)

In [ ]:
terbaik = YOLO(f'{SIMPAN}/v1/weights/best.pt')
m = terbaik.val(data=DATA, split='test', imgsz=1280, project=SIMPAN, name='v1_test', exist_ok=True)

print(f'mask mAP50    : {m.seg.map50:.3f}')
print(f'mask mAP50-95 : {m.seg.map:.3f}')
for i, c in enumerate(m.seg.ap_class_index):
    print(f'{m.names[int(c)]:6s} presisi={m.seg.p[i]:.3f}  recall={m.seg.r[i]:.3f}  mAP50={m.seg.ap50[i]:.3f}')

### 7. Uji tingkat lot: proporsi cacat sebenarnya vs terhitung, per varietas

In [ ]:
import pandas as pd

baris = []
for img in sorted(Path(ds.location, 'test', 'images').glob('*.jpg')):
    lbl = Path(ds.location, 'test', 'labels', img.stem + '.txt')
    benar = Counter(NAMES[int(b.split()[0])] for b in lbl.read_text().splitlines() if b.strip())
    r = terbaik.predict(str(img), imgsz=1280, conf=0.4, verbose=False)[0]
    pred = Counter(r.names[int(c)] for c in r.boxes.cls.tolist())
    baris.append(dict(berkas=img.name[:22], varietas=img.name.split('_')[0],
                      baik_benar=benar['baik'], cacat_benar=benar['cacat'],
                      baik_pred=pred['baik'], cacat_pred=pred['cacat']))

df = pd.DataFrame(baris)
df['prop_cacat_benar'] = (df.cacat_benar / (df.baik_benar + df.cacat_benar)).round(3)
df['prop_cacat_pred'] = (df.cacat_pred / (df.baik_pred + df.cacat_pred).clip(lower=1)).round(3)
print(df.to_string(index=False))
print()
print(df.groupby('varietas')[['baik_benar', 'baik_pred', 'cacat_benar', 'cacat_pred']].sum())

### 8. Lihat hasil prediksi satu foto

In [ ]:
from IPython.display import Image, display

contoh = sorted(Path(ds.location, 'test', 'images').glob('VA_*'))[0]
terbaik.predict(str(contoh), imgsz=1280, conf=0.4, save=True,
                project=SIMPAN, name='prediksi', exist_ok=True)
display(Image(filename=f'{SIMPAN}/prediksi/{contoh.name}', width=900))

### 9. Ekspor model untuk aplikasi SIGMA

In [ ]:
berkas = terbaik.export(format='onnx', imgsz=1280)
print('Tersimpan:', berkas)   # ikut tersimpan di Google Drive, di samping best.pt